# Lab 3: Search and A* — Using an LLM as an Engineering Assistant

## Objective

This laboratory focuses on formulating a warehouse navigation problem as a search problem and implementing A* search with the assistance of a Large Language Model (LLM).

The laboratory covers:
- Formulating a navigation problem as a search problem
- Identifying states, actions, transitions, goals, and costs
- Designing a search agent before implementation
- Using an LLM to assist in implementing A* search
- Testing and validating the generated program
- Comparing BFS and A*
- Investigating the effect of different heuristics
- Critically evaluating the use of an LLM as an engineering assistant

## Task 0: Formulate the Search Problem

The warehouse navigation problem can be represented as:

P = (S, A, T, s₀, G, c)

where:

| Component | Specification |
|---|---|
| State S | The current row and column position of the robot in the warehouse grid |
| Actions A | Move Up, Down, Left, or Right |
| Transition T | Moving the robot from its current cell to an adjacent valid cell |
| Initial state s₀ | The position marked `S` |
| Goal G | The position marked `G` |
| Cost c | 1 for every valid movement |

### (a) Information necessary to specify a state

A state can be specified by the robot's current `(row, column)` position in the warehouse grid. The warehouse layout and obstacle locations are part of the environment.

### (b) What makes an action invalid?

An action is invalid if it:
- moves the robot outside the warehouse grid; or
- moves the robot into an obstacle represented by `#`.

### (c) Is this a deterministic search problem?

Yes. Given the current state and a valid action, the resulting next state is uniquely determined.

### (d) What constitutes a solution?

A solution is a sequence of valid movements that takes the robot from the initial state `S` to the goal state `G` without passing through obstacles.

## Task 1: Plan the Search Agent

### 1. State Representation

Each state will be represented as a Python tuple:

`(row, column)`

For example, `(1, 1)` represents the cell in row 1 and column 1 of the grid.

### 2. Warehouse Representation

The warehouse will be represented as a two-dimensional Python list of characters.

For example:

- `#` → obstacle
- `.` → free cell
- `S` → starting position
- `G` → goal position

### 3. Determining Valid Actions

For each state, the agent will examine the four possible movements:

- Up: `(-1, 0)`
- Down: `(1, 0)`
- Left: `(0, -1)`
- Right: `(0, 1)`

A movement is valid if the resulting position is inside the grid and is not an obstacle.

### 4. Goal Test

The agent has reached the goal when the current state is equal to the position of `G`.

### 5. Frontier Information

The A* frontier should store states that have been discovered but not yet expanded. Each frontier entry should contain enough information to determine its priority using:

`f(n) = g(n) + h(n)`

where:

- `g(n)` is the cost from the start to the current state.
- `h(n)` is the estimated cost from the current state to the goal.

### 6. Path Reconstruction

The agent will store a parent relationship for each discovered state. When the goal is reached, the program will follow the parent relationships backwards from the goal to the start and then reverse the resulting sequence.

### 7. Information Reported by the Program

When the program terminates, it should report:

- Whether a solution was found
- The solution path
- The path length
- The number of states expanded

### Agent Design

The overall operation of the search agent is:

```text
Warehouse Grid
      ↓
Current State
      ↓
Generate Valid Actions
      ↓
Generate Successor States
      ↓
Calculate g(n), h(n), and f(n)
      ↓
A* Frontier
      ↓
Select State with Lowest f(n)
      ↓
Goal Test
   ↙        ↘
No          Yes
↓            ↓
Continue    Reconstruct Path
              ↓
          Report Results

## Task 2: LLM-Assisted A* Implementation
I am implementing a search agent in Python for a warehouse robot navigation problem.

The warehouse is represented as a 2D grid:

#################
#S....#.........#
#.###.#.#######.#
#...#.#.......#.#
###.#.#######.#.#
#...#.........#.#
#.###########.#.#
#.............#G#
#################

The symbols have the following meanings:
- S is the starting position.
- G is the goal position.
- # represents an obstacle.
- . represents a free cell.

The robot can move one cell at a time in four directions:
- Up
- Down
- Left
- Right

Every movement has a cost of 1.

Implement A* search according to the following design:

1. Represent each state as a (row, column) position.
2. Represent the warehouse as a two-dimensional Python list.
3. Determine valid actions by checking that the resulting position is inside the grid and is not an obstacle.
4. Recognise the goal when the current state is equal to the position of G.
5. Maintain a frontier that prioritises states using:
   f(n) = g(n) + h(n)
   where g(n) is the cost from the start to the current state and h(n) is the heuristic estimate to the goal.
6. Use Manhattan distance as the heuristic:
   h(n) = |row - goal_row| + |column - goal_column|
7. Store parent information so that the final path can be reconstructed from G back to S.
8. Avoid unnecessary repeated expansion of states.
9. Report:
   - whether a solution was found;
   - the solution path;
   - the path length;
   - the number of states expanded.
10. Print the warehouse with the discovered path marked if a solution exists.
11. Print a suitable message if no path exists.
12. Keep the implementation simple and suitable for an AI laboratory exercise.
13. Use only Python's standard library.
14. Include comments explaining the main components of the A* implementation.

After providing the code, briefly explain:
- how the frontier is implemented;
- how g(n), h(n), and f(n) are calculated;
- how states are prevented from being unnecessarily expanded repeatedly;
- how the path is reconstructed.

In [20]:
import heapq


# Warehouse represented as a 2D list
warehouse = [
    list("#################"),
    list("#S....#.........#"),
    list("#.###.#.#######.#"),
    list("#...#.#.......#.#"),
    list("###.#.#######.#.#"),
    list("#...#.........#.#"),
    list("#.###########.#.#"),
    list("#.............#G#"),
    list("#################")
]


# Four possible movements: Up, Down, Left, Right
DIRECTIONS = [
    (-1, 0),  # Up
    (1, 0),   # Down
    (0, -1),  # Left
    (0, 1)    # Right
]


# Find the start and goal positions
start = None
goal = None

for r in range(len(warehouse)):
    for c in range(len(warehouse[r])):
        if warehouse[r][c] == 'S':
            start = (r, c)
        elif warehouse[r][c] == 'G':
            goal = (r, c)


def heuristic(state):
    """Return Manhattan distance from state to the goal."""
    row, col = state
    goal_row, goal_col = goal

    return abs(row - goal_row) + abs(col - goal_col)


def get_valid_actions(state):
    """Return all valid neighbouring positions."""
    row, col = state
    valid_states = []

    for dr, dc in DIRECTIONS:
        new_row = row + dr
        new_col = col + dc

        # Check that the new position is inside the grid
        # and is not an obstacle.
        if (0 <= new_row < len(warehouse) and
                0 <= new_col < len(warehouse[0]) and
                warehouse[new_row][new_col] != '#'):

            valid_states.append((new_row, new_col))

    return valid_states


def reconstruct_path(parent, current):
    """Reconstruct path by following parent pointers back to S."""
    path = []

    while current is not None:
        path.append(current)
        current = parent[current]

    path.reverse()
    return path


def a_star():
    """
    Perform A* search from S to G.

    Returns:
        path: solution path, or None if no solution exists
        expanded: number of states expanded
    """

    # Priority queue storing (f(n), g(n), state)
    frontier = []

    # Cost from start to each state
    g_cost = {start: 0}

    # Parent information for path reconstruction
    parent = {start: None}

    # States that have already been expanded
    expanded_states = set()

    # Initially, f(start) = g(start) + h(start)
    initial_f = heuristic(start)
    heapq.heappush(frontier, (initial_f, 0, start))

    while frontier:

        # Get the state with the smallest f(n)
        f, current_g, current = heapq.heappop(frontier)

        # Ignore outdated entries in the priority queue
        if current_g != g_cost[current]:
            continue

        # Avoid expanding the same state unnecessarily
        if current in expanded_states:
            continue

        expanded_states.add(current)

        # Goal test
        if current == goal:
            path = reconstruct_path(parent, current)
            return path, len(expanded_states)

        # Explore neighbouring states
        for neighbour in get_valid_actions(current):

            # Every movement has cost 1
            new_g = current_g + 1

            # Only update if this is a better path to the neighbour
            if neighbour not in g_cost or new_g < g_cost[neighbour]:

                g_cost[neighbour] = new_g
                parent[neighbour] = current

                # f(n) = g(n) + h(n)
                new_f = new_g + heuristic(neighbour)

                heapq.heappush(
                    frontier,
                    (new_f, new_g, neighbour)
                )

    # No path was found
    return None, len(expanded_states)


def print_solution(path):
    """Print the warehouse with the discovered path marked."""

    # Mark the path using '*', keeping S and G unchanged
    for row, col in path:
        if warehouse[row][col] not in ('S', 'G'):
            warehouse[row][col] = '*'

    print("\nWarehouse with discovered path:")
    for row in warehouse:
        print("".join(row))


# Run A* search
path, states_expanded = a_star()

if path is not None:
    print("Solution found!")
    print("Solution path:")
    print(path)

    # Number of movements is one less than number of positions
    path_length = len(path) - 1

    print("Path length:", path_length)
    print("States expanded:", states_expanded)

    print_solution(path)

else:
    print("No path exists from S to G.")
    print("States expanded:", states_expanded)

Solution found!
Solution path:
[(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5), (5, 6), (5, 7), (5, 8), (5, 9), (5, 10), (5, 11), (5, 12), (5, 13), (4, 13), (3, 13), (3, 12), (3, 11), (3, 10), (3, 9), (3, 8), (3, 7), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (2, 15), (3, 15), (4, 15), (5, 15), (6, 15), (7, 15)]
Path length: 40
States expanded: 64

Warehouse with discovered path:
#################
#S****#*********#
#.###*#*#######*#
#...#*#*******#*#
###.#*#######*#*#
#...#*********#*#
#.###########.#*#
#.............#G#
#################


## Task 3: Test the Generated Program

### Test 1: Original Warehouse

The first LLM-generated A* program was run on the warehouse map supplied in the laboratory.

### Results

- **Solution found:** Yes
- **Path length:** 40 moves
- **States expanded:** 64
- **Goal reached:** Yes
- **Obstacles crossed:** No

The program successfully found a collision-free path from `S` at `(1, 1)` to `G` at `(7, 15)`.

### Solution Path

The path returned by the program was:

`[(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5), (5, 6), (5, 7), (5, 8), (5, 9), (5, 10), (5, 11), (5, 12), (5, 13), (4, 13), (3, 13), (3, 12), (3, 11), (3, 10), (3, 9), (3, 8), (3, 7), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (2, 15), (3, 15), (4, 15), (5, 15), (6, 15), (7, 15)]`

The resulting warehouse display showed the path marked with `*` while avoiding all obstacles.

### Observation

The generated A* implementation successfully solved the original warehouse problem. The program reported a path length of 40 moves and expanded 64 states.

In [21]:
# Test 2: Trivial case
# The goal is immediately adjacent to the start.

warehouse = [
    list("#####"),
    list("#SG##"),
    list("#####")
]

# Find the start and goal positions
start = None
goal = None

for r in range(len(warehouse)):
    for c in range(len(warehouse[r])):
        if warehouse[r][c] == 'S':
            start = (r, c)
        elif warehouse[r][c] == 'G':
            goal = (r, c)

# Run A* on the trivial map
path, states_expanded = a_star()

if path is not None:
    print("Solution found!")
    print("Solution path:", path)
    print("Path length:", len(path) - 1)
    print("States expanded:", states_expanded)
else:
    print("No path exists from S to G.")
    print("States expanded:", states_expanded)

Solution found!
Solution path: [(1, 1), (1, 2)]
Path length: 1
States expanded: 2


### Test 2: Trivial Case

A small warehouse was constructed in which the goal was immediately adjacent to the starting position.

#### Results

- **Solution found:** Yes
- **Solution path:** `[(1, 1), (1, 2)]`
- **Path length:** 1 move
- **States expanded:** 2

The program correctly found the one-step solution from `S` to `G`. This confirms that the A* implementation handles a simple case where the goal is directly reachable from the initial state.

In [22]:
# Test 3: No-solution case
# The goal is completely inaccessible.

warehouse = [
    list("#######"),
    list("#S....#"),
    list("###.###"),
    list("#...#G#"),
    list("#######")
]

# Find the start and goal positions
start = None
goal = None

for r in range(len(warehouse)):
    for c in range(len(warehouse[r])):
        if warehouse[r][c] == 'S':
            start = (r, c)
        elif warehouse[r][c] == 'G':
            goal = (r, c)

# Run A* on the no-solution map
path, states_expanded = a_star()

if path is not None:
    print("Solution found!")
    print("Solution path:", path)
    print("Path length:", len(path) - 1)
    print("States expanded:", states_expanded)
else:
    print("No path exists from S to G.")
    print("States expanded:", states_expanded)

No path exists from S to G.
States expanded: 9


### Test 3: No-Solution Case

A warehouse was constructed in which the goal was completely inaccessible from the starting position.

#### Results

- **Solution found:** No
- **Path:** No path exists from `S` to `G`
- **States expanded:** 9

The program correctly detected that no path exists and terminated normally. It did not enter an infinite loop.

In [23]:
# Test 4: Alternative paths
# The map contains multiple possible paths from S to G.

warehouse = [
    list("#######"),
    list("#S...G#"),
    list("#.....#"),
    list("#######")
]

# Find the start and goal positions
start = None
goal = None

for r in range(len(warehouse)):
    for c in range(len(warehouse[r])):
        if warehouse[r][c] == 'S':
            start = (r, c)
        elif warehouse[r][c] == 'G':
            goal = (r, c)

# Run A* on the alternative-path map
path, states_expanded = a_star()

if path is not None:
    print("Solution found!")
    print("Solution path:", path)
    print("Path length:", len(path) - 1)
    print("States expanded:", states_expanded)
else:
    print("No path exists from S to G.")
    print("States expanded:", states_expanded)

Solution found!
Solution path: [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5)]
Path length: 4
States expanded: 5


### Test 4: Alternative Paths

A warehouse containing multiple possible paths from `S` to `G` was used to test whether A* returns a shortest path.

#### Results

- **Solution found:** Yes
- **Solution path:** `[(1, 1), (1, 2), (1, 3), (1, 4), (1, 5)]`
- **Path length:** 4 moves
- **States expanded:** 5

The returned path moves directly from `S` to `G` along the upper row. Since the start and goal are four columns apart, 4 moves is the shortest possible path length.

Therefore, the test confirms that the A* implementation returned a shortest path for this alternative-path case.

## Task 4: Inspect the A* Algorithm

The LLM-generated program can be mapped to the components of the search problem as follows:

| Concept | Where it appears in the code |
|---|---|
| **State** | A state is represented as a `(row, column)` tuple, such as `(1, 1)`. |
| **Action** | The four movements are represented by `DIRECTIONS = [(-1,0), (1,0), (0,-1), (0,1)]`. |
| **Transition** | `get_valid_actions(state)` generates valid neighbouring states after applying an action. |
| **Goal test** | `if current == goal:` inside `a_star()`. |
| **g(n)** | `g_cost` stores the cost from the start, and `new_g = current_g + 1` calculates the cost of reaching a neighbour. |
| **h(n)** | The `heuristic(state)` function calculates Manhattan distance to the goal. |
| **f(n)** | `new_f = new_g + heuristic(neighbour)` explicitly calculates `f(n) = g(n) + h(n)`. |
| **Frontier** | `frontier` is implemented using Python's `heapq` priority queue. |
| **Visited states** | `expanded_states` stores states that have already been expanded. |
| **Path reconstruction** | `reconstruct_path(parent, current)` follows parent pointers from the goal back to the start. |

### Questions

**(a) What data structure is used for the A* frontier?**

The A* frontier is implemented using a priority queue provided by Python's `heapq` module. The entries contain `(f(n), g(n), state)`, so the state with the smallest `f(n)` is selected first.

**(b) How does the program select the next state to expand?**

The program uses `heapq.heappop(frontier)` to remove the entry with the smallest `f(n)` from the priority queue. This state is then selected for expansion.

**(c) Where is the heuristic calculated?**

The heuristic is calculated in the `heuristic(state)` function using Manhattan distance:

`h(n) = |row - goal_row| + |column - goal_column|`

**(d) Does the program explicitly calculate f(n) = g(n) + h(n)?**

Yes. The program explicitly calculates:

`new_f = new_g + heuristic(neighbour)`

This corresponds directly to:

`f(n) = g(n) + h(n)`

**(e) How does the program prevent unnecessary repeated exploration?**

The program maintains an `expanded_states` set. Before expanding a state, it checks whether the state has already been expanded. It also compares the `current_g` value with the stored `g_cost` to ignore outdated entries in the priority queue.

In [24]:
# Task 5: Compare BFS with A*

from collections import deque

# Restore the original warehouse
warehouse = [
    list("#################"),
    list("#S....#.........#"),
    list("#.###.#.#######.#"),
    list("#...#.#.......#.#"),
    list("###.#.#######.#.#"),
    list("#...#.........#.#"),
    list("#.###########.#.#"),
    list("#.............#G#"),
    list("#################")
]

# Find start and goal
start = None
goal = None

for r in range(len(warehouse)):
    for c in range(len(warehouse[r])):
        if warehouse[r][c] == 'S':
            start = (r, c)
        elif warehouse[r][c] == 'G':
            goal = (r, c)


def bfs():
    """Perform Breadth-First Search from S to G."""

    queue = deque([start])
    visited = {start}
    parent = {start: None}
    expanded = 0

    while queue:
        current = queue.popleft()
        expanded += 1

        # Goal test
        if current == goal:
            path = reconstruct_path(parent, current)
            return path, expanded

        for neighbour in get_valid_actions(current):

            if neighbour not in visited:
                visited.add(neighbour)
                parent[neighbour] = current
                queue.append(neighbour)

    return None, expanded


# Run BFS
bfs_path, bfs_expanded = bfs()

# Run A*
astar_path, astar_expanded = a_star()

print("BFS Results")
print("Solution found:", bfs_path is not None)

if bfs_path is not None:
    print("Path length:", len(bfs_path) - 1)
else:
    print("Path length: N/A")

print("States expanded:", bfs_expanded)

print("\nA* Results")
print("Solution found:", astar_path is not None)

if astar_path is not None:
    print("Path length:", len(astar_path) - 1)
else:
    print("Path length: N/A")

print("States expanded:", astar_expanded)

BFS Results
Solution found: True
Path length: 40
States expanded: 64

A* Results
Solution found: True
Path length: 40
States expanded: 64


### Task 5: BFS vs A* Comparison

Both BFS and A* were run on the same original warehouse map.

| Measure | BFS | A* |
|---|---:|---:|
| Solution found | Yes | Yes |
| Path length | 40 | 40 |
| States expanded | 64 | 64 |

### Observations

Both algorithms successfully found a solution.

Both algorithms returned a path of length 40 moves. Therefore, the solution path lengths were the same.

For this particular warehouse, both algorithms expanded 64 states. Thus, A* did not expand fewer states than BFS in this experiment.

A* uses additional information through its heuristic, while BFS explores states according to their distance from the start. However, the benefit of the heuristic depends on the structure of the search problem and the resulting search order. For this particular map, the measured number of expanded states was the same for both algorithms.

In [25]:
# Task 6: Investigating Different Heuristics

import math
import heapq

warehouse = [
    list("#################"),
    list("#S....#.........#"),
    list("#.###.#.#######.#"),
    list("#...#.#.......#.#"),
    list("###.#.#######.#.#"),
    list("#...#.........#.#"),
    list("#.###########.#.#"),
    list("#.............#G#"),
    list("#################")
]

start = None
goal = None

for r in range(len(warehouse)):
    for c in range(len(warehouse[r])):
        if warehouse[r][c] == 'S':
            start = (r, c)
        elif warehouse[r][c] == 'G':
            goal = (r, c)


def get_valid_actions(state):
    row, col = state
    valid_states = []

    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        new_row = row + dr
        new_col = col + dc

        if (0 <= new_row < len(warehouse) and
                0 <= new_col < len(warehouse[0]) and
                warehouse[new_row][new_col] != '#'):
            valid_states.append((new_row, new_col))

    return valid_states


def reconstruct_path(parent, current):
    path = []

    while current is not None:
        path.append(current)
        current = parent[current]

    path.reverse()
    return path


# Heuristic 1: Manhattan distance
def manhattan(state):
    row, col = state
    goal_row, goal_col = goal
    return abs(row - goal_row) + abs(col - goal_col)


# Heuristic 2: h(n) = 0
def zero_heuristic(state):
    return 0


# Heuristic 3: Euclidean distance
def euclidean(state):
    row, col = state
    goal_row, goal_col = goal
    return math.sqrt((row - goal_row) ** 2 + (col - goal_col) ** 2)


# Heuristic 4: 2 × Manhattan distance
def double_manhattan(state):
    return 2 * manhattan(state)


def a_star_with_heuristic(heuristic_function):
    frontier = []
    g_cost = {start: 0}
    parent = {start: None}
    expanded_states = set()

    initial_f = heuristic_function(start)
    heapq.heappush(frontier, (initial_f, 0, start))

    while frontier:
        f, current_g, current = heapq.heappop(frontier)

        if current_g != g_cost[current]:
            continue

        if current in expanded_states:
            continue

        expanded_states.add(current)

        if current == goal:
            path = reconstruct_path(parent, current)
            return path, len(expanded_states)

        for neighbour in get_valid_actions(current):
            new_g = current_g + 1

            if neighbour not in g_cost or new_g < g_cost[neighbour]:
                g_cost[neighbour] = new_g
                parent[neighbour] = current

                new_f = new_g + heuristic_function(neighbour)

                heapq.heappush(
                    frontier,
                    (new_f, new_g, neighbour)
                )

    return None, len(expanded_states)


# Run all four heuristics
heuristics = [
    ("Manhattan", manhattan),
    ("h(n) = 0", zero_heuristic),
    ("Euclidean", euclidean),
    ("2 × Manhattan", double_manhattan)
]

for name, heuristic_function in heuristics:
    path, states_expanded = a_star_with_heuristic(heuristic_function)

    print(name)
    print("Solution found:", path is not None)

    if path is not None:
        print("Path length:", len(path) - 1)
    else:
        print("Path length: N/A")

    print("States expanded:", states_expanded)
    print()

Manhattan
Solution found: True
Path length: 40
States expanded: 64

h(n) = 0
Solution found: True
Path length: 40
States expanded: 64

Euclidean
Solution found: True
Path length: 40
States expanded: 64

2 × Manhattan
Solution found: True
Path length: 40
States expanded: 64



## Task 6: Heuristic Investigation

The A* algorithm was tested using four different heuristic functions on the same warehouse.

| Heuristic | Solution Found | Path Length | States Expanded |
|---|---|---:|---:|
| Manhattan distance | Yes | 40 | 64 |
| h(n) = 0 | Yes | 40 | 64 |
| Euclidean distance | Yes | 40 | 64 |
| 2 × Manhattan distance | Yes | 40 | 64 |

### Observations

All four heuristic choices found a solution with a path length of 40 moves.

For this particular warehouse and implementation, all four heuristics also resulted in 64 expanded states. Therefore, no difference in the number of expanded states was observed in this experiment.

The heuristic h(n) = 0 provides no additional information about the distance to the goal. In this case, A* effectively behaves like a uniform-cost search, and because every movement has cost 1, this is closely related to BFS.

Manhattan distance estimates the remaining number of horizontal and vertical moves needed to reach the goal. It is suitable for this four-directional grid because each move has a cost of 1 and there are no diagonal movements.

Euclidean distance is another estimate of the distance to the goal. It is generally smaller than or equal to the Manhattan distance for the same state.

The 2 × Manhattan heuristic is more aggressive because it multiplies the Manhattan estimate by 2. Unlike the ordinary Manhattan heuristic, this heuristic is not admissible in general because it can overestimate the actual remaining cost.

Despite these theoretical differences, the experiment produced the same measured path length and number of expanded states for all four heuristics on this particular map.

## Task 7: Evaluation of the LLM-Generated Agent

### What did the LLM get correct immediately?

The LLM correctly implemented the main components of A* search:
- States were represented as `(row, column)` positions.
- Valid neighbouring states were generated using the four possible movements.
- Manhattan distance was used as the heuristic.
- The frontier was implemented using Python's `heapq` priority queue.
- The priority was based on `f(n) = g(n) + h(n)`.
- Parent information was stored for path reconstruction.
- The program detected when the goal was reached and reported the path, path length, and states expanded.

### What bugs or design issues did you discover?

The generated program did not contain a major functional bug during the tests performed. However, the implementation was treated as a hypothesis rather than assumed to be correct.

One design detail was that the warehouse and search functions used global variables. This works for the laboratory experiments, but a more reusable implementation could pass the warehouse, start, and goal as function parameters.

Another detail is that `print_solution()` modifies the warehouse by replacing path cells with `*`. Therefore, the original warehouse must be restored before running another experiment on the same grid.

### What did you learn from inspecting the generated code?

Inspecting the code helped connect the theoretical A* concepts to their implementation. In particular:
- `g_cost` stores the current cost from the start.
- `heuristic()` calculates `h(n)`.
- The priority queue stores states using their `f(n)` value.
- `expanded_states` prevents states from being expanded repeatedly.
- `parent` stores the predecessor of each state and allows the final path to be reconstructed.

### What terminology did you need to understand?

The main terms that needed to be connected to the code were:
- State
- Action
- Transition
- Frontier
- `g(n)`
- `h(n)`
- `f(n)`
- Expanded/visited states
- Path reconstruction
- Admissible heuristic

### What did you test?

The generated program was tested using:
1. The original warehouse.
2. A one-step solution.
3. A warehouse with no possible solution.
4. A warehouse containing alternative paths.
5. A comparison between BFS and A*.
6. Four different heuristic functions.

These tests checked both successful and unsuccessful search cases and allowed the behaviour of the algorithm to be compared experimentally.

### What did you change or verify?

The overall search design was decided before asking the LLM for code. The LLM was then asked to implement that design.

The generated program was run and its behaviour was verified using the required test cases. The experimental results were recorded rather than assuming that the generated code was correct.

### Would you trust the LLM-generated code without testing?

No. The LLM-generated code should be treated as a proposed implementation rather than automatically correct software. Testing was necessary to verify that it found valid paths, handled a one-step problem, terminated when no solution existed, and behaved as expected when compared with BFS and different heuristics.

# Final Reflection

### 1. Why is problem formulation important before writing code?

Problem formulation is important because it clearly defines the components of the search problem: states, actions, transitions, initial state, goal state, and cost. It gives a precise description of what the agent needs to solve before implementation begins. In this laboratory, representing the robot's position as a state and the four possible movements as actions made the warehouse navigation problem suitable for search algorithms.

### 2. Why is A* considered an informed search algorithm?

A* is an informed search algorithm because it uses a heuristic function to estimate the remaining cost from the current state to the goal. It combines the actual cost already travelled, `g(n)`, with the estimated remaining cost, `h(n)`, using:

`f(n) = g(n) + h(n)`

The heuristic provides additional information about the direction of the goal compared with an uninformed method such as BFS.

### 3. How does the choice of heuristic affect A*?

The heuristic affects the order in which A* explores states. A useful heuristic can guide the search towards the goal more effectively. For the warehouse problem, Manhattan distance is suitable because the robot moves only horizontally or vertically and every movement has cost 1.

In the experiments, Manhattan distance, `h(n) = 0`, Euclidean distance, and `2 × Manhattan distance` all produced a path length of 40 and 64 expanded states. Therefore, no difference was observed for these measurements on this particular warehouse.

However, the heuristics have different theoretical properties. Manhattan distance is admissible for this grid, while `2 × Manhattan distance` can overestimate the remaining cost and is therefore not generally admissible.

### 4. What contribution did the LLM make?

The LLM was used as an engineering assistant to generate an implementation of the A* algorithm based on a design that had already been specified. It provided the Python implementation of the frontier, heuristic calculation, cost tracking, state expansion, and path reconstruction.

The generated code was then tested using multiple test cases rather than being accepted without verification. This helped demonstrate how an LLM can accelerate implementation while the engineer remains responsible for checking the resulting program.

### 5. What are the risks of using LLM-generated code without testing?

LLM-generated code may contain bugs, incorrect assumptions, or design choices that are not appropriate for a particular problem. Code can appear correct while failing on edge cases such as an unsolvable map or a different input structure.

In this laboratory, testing was therefore an important part of the engineering process. The A* implementation was tested on the original warehouse, a one-step problem, an unsolvable problem, alternative paths, BFS comparison, and different heuristic functions.

The main lesson is that an LLM can assist with implementation, but the engineer must understand the problem, evaluate the generated code, test its behaviour, and take responsibility for the final result.

## Final A* Implementation

The following implementation was generated with LLM assistance and validated using the test cases described in Task 3.